In [1]:
import sqlite3
import os

os.makedirs("data/databases",exist_ok=True)

In [2]:
conn = sqlite3.connect('data/databases/school.db')
cursor = conn.cursor()

In [3]:
cursor.execute('''CREATE TABLE Students (
               StudentID VARCHAR(10) PRIMARY KEY,
               StudentName VARCHAR(50),
               Class INT,
               Age INT
               )''')

In [4]:
cursor.execute('''CREATE TABLE Subjects(
               SubjectID VARCHAR(10) PRIMARY KEY,
               SubjectName VARCHAR(50)
               )''')

In [5]:
cursor.execute('''CREATE TABLE Marks(
               StudentID VARCHAR(10),
               SubjectID VARCHAR(10),
               Marks INT,
               FOREIGN KEY (StudentID) REFERENCES Students(StudentID),
               FOREIGN KEY (SubjectID) REFERENCES Subjects(SubjectID)
               )''')

In [6]:
students = [
    ('S001', 'Ananya Sharma', 10, 15),
    ('S002', 'Rahul Verma', 10, 16),
    ('S003', 'Sai Pavan', 10, 15),
    ('S004', 'Meera Reddy', 10, 16),
    ('S005', 'Arjun Patel', 10, 15)
]

subjects = [
    ('SUB01','Mathematics'),
    ('SUB02', 'Science'),
    ('SUB03', 'English'),
    ('SUB04', 'Social Studies'),
    ('SUB05', 'Computer Science')
]

marks = [
    ('S001', 'SUB01',89),
    ('S001', 'SUB02', 76),
    ('S002', 'SUB01', 92),
    ('S002', 'SUB03', 81),
    ('S003', 'SUB05', 95),
    ('S004', 'SUB04', 73),
    ('S005', 'SUB02', 88)
]

In [7]:
cursor.executemany('''INSERT INTO Students VALUES (?,?,?,?)''', students)
cursor.executemany('''INSERT INTO Subjects VALUES (?,?)''', subjects)
cursor.executemany('''INSERT INTO Marks VALUES (?,?,?)''', marks)

In [8]:
cursor.execute("SELECT * FROM Students")

In [9]:
conn.commit()
conn.close()

In [10]:
from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader

In [11]:
db = SQLDatabase.from_uri("sqlite:///data/databases/school.db")

print(f"Tables: {db.get_usable_table_names()}")
print("\nTables")
print(db.get_table_info())

Tables: ['Marks', 'Students', 'Subjects']

Tables

CREATE TABLE "Marks" (
	"StudentID" VARCHAR(10), 
	"SubjectID" VARCHAR(10), 
	"Marks" INTEGER, 
	FOREIGN KEY("StudentID") REFERENCES "Students" ("StudentID"), 
	FOREIGN KEY("SubjectID") REFERENCES "Subjects" ("SubjectID")
)

/*
3 rows from Marks table:
StudentID	SubjectID	Marks
S001	SUB01	89
S001	SUB02	76
S002	SUB01	92
*/


CREATE TABLE "Students" (
	"StudentID" VARCHAR(10), 
	"StudentName" VARCHAR(50), 
	"Class" INTEGER, 
	"Age" INTEGER, 
	PRIMARY KEY ("StudentID")
)

/*
3 rows from Students table:
StudentID	StudentName	Class	Age
S001	Ananya Sharma	10	15
S002	Rahul Verma	10	16
S003	Sai Pavan	10	15
*/


CREATE TABLE "Subjects" (
	"SubjectID" VARCHAR(10), 
	"SubjectName" VARCHAR(50), 
	PRIMARY KEY ("SubjectID")
)

/*
3 rows from Subjects table:
SubjectID	SubjectName
SUB01	Mathematics
SUB02	Science
SUB03	English
*/


In [12]:
from typing import List
from langchain_core.documents import Document

print("SQL Processing")

def sql_to_documents(db_path: str)->List[Document]:
    """"Convert SQL Database To documents with context"""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    documents = []

    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    for table in tables:
        table_name = table[0]

        cursor.execute(f"PRAGMA table_info({table_name});")
        columns = cursor.fetchall()
        column_names = [col[1] for col in columns]

        cursor.execute(f"SELECT * FROM {table_name}")
        rows = cursor.fetchall()

        table_content = f"Table : {table_name}\n"
        table_content += f"Columns : {', '.join(column_names)}\n"
        table_content += f"Total Records: {len(rows)}\n\n"

        table_content += "Sample Records:\n"
        for row in rows[:5]:
            record = dict(zip(column_names,row))
            table_content += f"{record}\n"

        doc = Document(
            page_content=table_content,
            metadata={
                'source': db_path,
                'table_name': table_name,
                'num_records': len(rows),
                'data_type': 'sql_table'
            }
        )
        documents.append(doc)
    
    cursor.execute("""
        SELECT s.StudentID, s.StudentName, m.Marks
        FROM Students as s
        JOIN Marks AS m ON s.StudentID = m.StudentID
    """)

    relationships = cursor.fetchall()
    rel_content = "Student-Marks Relationships:\n\n"

    for rel in relationships:
        rel_content += f"StudentId : {rel[0]}, Name: {rel[1]}, Marks: {rel[2]}\n"

    rel_doc = Document(
        page_content=rel_content,
        metadata = {
            'source': db_path,
            'data_type': 'sql_relationships',
            'query': 'students_marks_join'
        }
    )
    documents.append(rel_doc)

    conn.close()
    return documents

SQL Processing


In [21]:
import sqlite3
import requests

def generate_sql_query_llama3(user_query, table_info):
    import json
    url = "http://localhost:11434/api/chat"
    prompt = (
        "You are an expert SQL assistant. "
        "Given the following database schema:\n"
        f"{table_info}\n"
        "Write an SQLite SQL query (do not explain, only output the SQL) for this question:\n"
        f"{user_query}"
    )
    payload = {
        "model": "llama3",
        "messages": [
            {"role": "system", "content": "You are an expert SQL assistant."},
            {"role": "user", "content": prompt}
        ],
        "stream": True  # Important: request streaming response
    }
    response = requests.post(url, json=payload, stream=True)
    sql_query = ""
    for line in response.iter_lines():
        if line:
            chunk = json.loads(line.decode("utf-8"))
            sql_query += chunk.get("message", {}).get("content", "")
    # Clean up code block formatting if present
    sql_query = sql_query.strip().strip("```sql").strip("```").strip()
    return sql_query

def execute_sql_query(db_path, sql_query):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    try:
        cursor.execute(sql_query)
        rows = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description] if cursor.description else []
        conn.close()
        return columns, rows
    except Exception as e:
        conn.close()
        return [], [f"SQL Error: {e}"]

# Usage example:
db_path = "data/databases/school.db"
from langchain_community.utilities import SQLDatabase
db = SQLDatabase.from_uri(f"sqlite:///{db_path}")
table_info = db.get_table_info()

user_query = "List all students who scored above 90 in any subject."
sql_query = generate_sql_query_llama3(user_query, table_info)
print("Generated SQL Query:\n", sql_query)

columns, rows = execute_sql_query(db_path, sql_query)
print("Results:")
print(columns)
for row in rows:
    print(row)

Generated SQL Query:
 SELECT s."StudentID", s."StudentName"
FROM "Students" s
WHERE s."StudentID" IN (
  SELECT m."StudentID"
  FROM "Marks" m
  WHERE m."Marks" > 90
)
Results:
['StudentID', 'StudentName']
('S002', 'Rahul Verma')
('S003', 'Sai Pavan')


In [22]:
# --- RAG + Text-to-SQL Hybrid Cell ---

from sentence_transformers import SentenceTransformer
import numpy as np
import requests
import sqlite3
import json

# 1. Load embedding model and prepare docs/chunks
model = SentenceTransformer("BAAI/bge-base-en-v1.5")
docs = sql_to_documents(db_path="data/databases/school.db")
texts = [doc.page_content for doc in docs]
embeddings = model.encode(texts, show_progress_bar=True)

def get_top_chunks(query, embeddings, texts, top_k=2):
    query_emb = model.encode([query])[0]
    similarities = np.dot(embeddings, query_emb) / (
        np.linalg.norm(embeddings, axis=1) * np.linalg.norm(query_emb) + 1e-8
    )
    top_indices = similarities.argsort()[-top_k:][::-1]
    return [texts[i] for i in top_indices]

def generate_sql_query_llama3(user_query, table_info):
    url = "http://localhost:11434/api/chat"
    prompt = (
        "You are an expert SQL assistant. "
        "Given the following database schema:\n"
        f"{table_info}\n"
        "Write an SQLite SQL query (do not explain, only output the SQL) for this question:\n"
        f"{user_query}"
    )
    payload = {
        "model": "llama3",
        "messages": [
            {"role": "system", "content": "You are an expert SQL assistant."},
            {"role": "user", "content": prompt}
        ],
        "stream": True
    }
    response = requests.post(url, json=payload, stream=True)
    sql_query = ""
    for line in response.iter_lines():
        if line:
            chunk = json.loads(line.decode("utf-8"))
            sql_query += chunk.get("message", {}).get("content", "")
    sql_query = sql_query.strip().strip("```sql").strip("```").strip()
    return sql_query

def execute_sql_query(db_path, sql_query):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    try:
        cursor.execute(sql_query)
        rows = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description] if cursor.description else []
        conn.close()
        return columns, rows
    except Exception as e:
        conn.close()
        return [], [f"SQL Error: {e}"]

# --- Hybrid Usage Example ---
db_path = "data/databases/school.db"
from langchain_community.utilities import SQLDatabase
db = SQLDatabase.from_uri(f"sqlite:///{db_path}")
table_info = db.get_table_info()

user_query = "List all students who scored above 90 in any subject."

# 1. RAG: Retrieve relevant context chunks
top_chunks = get_top_chunks(user_query, embeddings, texts, top_k=2)
context = "\n\n".join(top_chunks)
print("RAG Context:\n", context)

# 2. Text-to-SQL: Generate SQL using Llama3 and schema
sql_query = generate_sql_query_llama3(user_query, table_info)
print("\nGenerated SQL Query:\n", sql_query)

# 3. Execute SQL and show results
columns, rows = execute_sql_query(db_path, sql_query)
print("\nResults:")
print(columns)
for row in rows:
    print(row)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 762.81it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1/1 [00:00<00:00,  1.66it/s]


RAG Context:
 Table : Students
Columns : StudentID, StudentName, Class, Age
Total Records: 5

Sample Records:
{'StudentID': 'S001', 'StudentName': 'Ananya Sharma', 'Class': 10, 'Age': 15}
{'StudentID': 'S002', 'StudentName': 'Rahul Verma', 'Class': 10, 'Age': 16}
{'StudentID': 'S003', 'StudentName': 'Sai Pavan', 'Class': 10, 'Age': 15}
{'StudentID': 'S004', 'StudentName': 'Meera Reddy', 'Class': 10, 'Age': 16}
{'StudentID': 'S005', 'StudentName': 'Arjun Patel', 'Class': 10, 'Age': 15}


Student-Marks Relationships:

StudentId : S001, Name: Ananya Sharma, Marks: 89
StudentId : S001, Name: Ananya Sharma, Marks: 76
StudentId : S002, Name: Rahul Verma, Marks: 92
StudentId : S002, Name: Rahul Verma, Marks: 81
StudentId : S003, Name: Sai Pavan, Marks: 95
StudentId : S004, Name: Meera Reddy, Marks: 73
StudentId : S005, Name: Arjun Patel, Marks: 88


Generated SQL Query:
 SELECT StudentName 
FROM Students 
WHERE StudentID IN (
  SELECT StudentID 
  FROM Marks 
  WHERE Marks > 90
);

Results:
[

In [25]:
# --- RAG + Text-to-SQL Hybrid Cell ---

from sentence_transformers import SentenceTransformer
import numpy as np
import requests
import sqlite3
import json

# 1. Load embedding model and prepare docs/chunks
model = SentenceTransformer("BAAI/bge-base-en-v1.5")
docs = sql_to_documents(db_path="data/databases/school.db")
texts = [doc.page_content for doc in docs]
embeddings = model.encode(texts, show_progress_bar=True)

def get_top_chunks(query, embeddings, texts, top_k=2):
    query_emb = model.encode([query])[0]
    similarities = np.dot(embeddings, query_emb) / (
        np.linalg.norm(embeddings, axis=1) * np.linalg.norm(query_emb) + 1e-8
    )
    top_indices = similarities.argsort()[-top_k:][::-1]
    return [texts[i] for i in top_indices]

def generate_sql_query_llama3(user_query, table_info):
    url = "http://localhost:11434/api/chat"
    prompt = (
        "You are an expert SQL assistant. "
        "Given the following database schema:\n"
        f"{table_info}\n"
        "Write an SQLite SQL query (do not explain, only output the SQL) for this question:\n"
        f"{user_query}"
    )
    payload = {
        "model": "llama3",
        "messages": [
            {"role": "system", "content": "You are an expert SQL assistant."},
            {"role": "user", "content": prompt}
        ],
        "stream": True
    }
    response = requests.post(url, json=payload, stream=True)
    sql_query = ""
    for line in response.iter_lines():
        if line:
            chunk = json.loads(line.decode("utf-8"))
            sql_query += chunk.get("message", {}).get("content", "")
    sql_query = sql_query.strip().strip("```sql").strip("```").strip()
    return sql_query

def execute_sql_query(db_path, sql_query):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    try:
        cursor.execute(sql_query)
        rows = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description] if cursor.description else []
        conn.close()
        return columns, rows
    except Exception as e:
        conn.close()
        return [], [f"SQL Error: {e}"]

# --- Hybrid Usage Example ---
db_path = "data/databases/school.db"
from langchain_community.utilities import SQLDatabase
db = SQLDatabase.from_uri(f"sqlite:///{db_path}")
table_info = db.get_table_info()

user_query = "List all students who scored above 90 in any subject."

# 1. RAG: Retrieve relevant context chunks
top_chunks = get_top_chunks(user_query, embeddings, texts, top_k=2)
context = "\n\n".join(top_chunks)
print("RAG Context:\n", context)

# 2. Text-to-SQL: Generate SQL using Llama3 and schema
sql_query = generate_sql_query_llama3(user_query, table_info)
print("\nGenerated SQL Query:\n", sql_query)

# 3. Execute SQL and show results
columns, rows = execute_sql_query(db_path, sql_query)
print("\nResults:")
print(columns)
for row in rows:
    print(row)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 602.87it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1/1 [00:01<00:00,  1.01s/it]


RAG Context:
 Table : Students
Columns : StudentID, StudentName, Class, Age
Total Records: 5

Sample Records:
{'StudentID': 'S001', 'StudentName': 'Ananya Sharma', 'Class': 10, 'Age': 15}
{'StudentID': 'S002', 'StudentName': 'Rahul Verma', 'Class': 10, 'Age': 16}
{'StudentID': 'S003', 'StudentName': 'Sai Pavan', 'Class': 10, 'Age': 15}
{'StudentID': 'S004', 'StudentName': 'Meera Reddy', 'Class': 10, 'Age': 16}
{'StudentID': 'S005', 'StudentName': 'Arjun Patel', 'Class': 10, 'Age': 15}


Student-Marks Relationships:

StudentId : S001, Name: Ananya Sharma, Marks: 89
StudentId : S001, Name: Ananya Sharma, Marks: 76
StudentId : S002, Name: Rahul Verma, Marks: 92
StudentId : S002, Name: Rahul Verma, Marks: 81
StudentId : S003, Name: Sai Pavan, Marks: 95
StudentId : S004, Name: Meera Reddy, Marks: 73
StudentId : S005, Name: Arjun Patel, Marks: 88


Generated SQL Query:
 SELECT * 
FROM "Students" 
WHERE "StudentID" IN (
    SELECT "StudentID" 
    FROM "Marks" 
    WHERE "Marks" > 90
);

Resu